# Version 2 — Réseau profond, régularisation, interprétabilité
**Responsable : FALL Madjiguène** · relecture : DIOP Alioune

Parties 3, 4, 5, 6 et 7 du sujet. Ce notebook est **autonome** : il ne demande que `train.csv` et `test.csv`
(dans `data/HAR/`, à côté du notebook ou dans le dossier parent) ; tout le code est défini ci-dessous.

**Temps de calcul.** Les expériences de la partie 5 (P3, P4, P7 : plus de 200 entraînements de 30 époques) prennent
environ 1 h 30 sur CPU. Chaque expérience enregistre ses résultats dans `results/` ; à l'exécution suivante, le
notebook les relit au lieu de tout recalculer (`RECALCULER = True` pour forcer le recalcul). Les graines sont fixées :
relire ou recalculer donne les mêmes chiffres.

## 1. Imports

In [ ]:
import copy, json, random, time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, silhouette_score
from sklearn.neighbors import NearestNeighbors
from sklearn.manifold import TSNE

def set_seed(seed=0):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
set_seed(0)
print("PyTorch", torch.__version__, "| device :", DEVICE)

# Dossier du projet : celui qui contient data/HAR (le dossier du notebook ou son parent)
ROOT = Path(".") if Path("data/HAR/train.csv").exists() else Path("..")
DATA_DIR, RESULTS_DIR = ROOT / "data" / "HAR", ROOT / "results"
LOG_DIR, FIG_DIR = RESULTS_DIR / "logs", RESULTS_DIR / "figures"
LOG_DIR.mkdir(parents=True, exist_ok=True); FIG_DIR.mkdir(parents=True, exist_ok=True)

CLASSES = ["WALKING", "WALKING_UPSTAIRS", "WALKING_DOWNSTAIRS", "SITTING", "STANDING", "LAYING"]
SEEDS = [0, 1, 2]           # graines de toutes les comparaisons répétées
EPOCHS = 30
RECALCULER = False          # True : refait toutes les expériences de la partie 5 (≈ 1 h 30)

## 2. Chargement et exploration des données
L'exploration détaillée est faite dans la version 1 ; on vérifie ici la taille, les classes et les volontaires.

In [ ]:
df = pd.read_csv(DATA_DIR / "train.csv")
features = [c for c in df.columns if c not in ("subject", "Activity")]
print(df.shape, "|", len(features), "variables |", df["subject"].nunique(), "volontaires")
df["Activity"].value_counts()

## 3. Découpage apprentissage / validation et DataLoaders
Même découpage que la version 1 : 5 volontaires **entiers** en validation (tirage de graine 42), puis
standardisation avec les statistiques de l'apprentissage uniquement.

In [ ]:
rng = np.random.default_rng(42)
val_subjects = sorted(rng.choice(np.sort(df["subject"].unique()), size=5, replace=False).tolist())
df_tr = df[~df["subject"].isin(val_subjects)].reset_index(drop=True)
df_va = df[df["subject"].isin(val_subjects)].reset_index(drop=True)
print("volontaires de validation :", val_subjects)
assert set(df_tr["subject"]) & set(df_va["subject"]) == set()

label_to_int = {c: i for i, c in enumerate(CLASSES)}
def to_xy(d):
    return d[features].to_numpy(dtype=np.float32), d["Activity"].map(label_to_int).to_numpy(dtype=np.int64)

X_tr_raw, y_tr = to_xy(df_tr)
X_va_raw, y_va = to_xy(df_va)
scaler = StandardScaler().fit(X_tr_raw)                       # apprentissage UNIQUEMENT
X_tr = scaler.transform(X_tr_raw).astype(np.float32)
X_va = scaler.transform(X_va_raw).astype(np.float32)
IN_DIM = X_tr.shape[1]

def make_loaders(Xtr, ytr, Xva, yva, batch_size=64, seed=0):
    g = torch.Generator().manual_seed(seed)                    # ordre des mini-lots reproductible
    tr_ds = TensorDataset(torch.as_tensor(Xtr, dtype=torch.float32), torch.as_tensor(ytr))
    va_ds = TensorDataset(torch.as_tensor(Xva, dtype=torch.float32), torch.as_tensor(yva))
    return (DataLoader(tr_ds, batch_size=batch_size, shuffle=True, generator=g),
            DataLoader(va_ds, batch_size=512, shuffle=False))

X_tr.shape, X_va.shape

## 4. Modèle et fonctions d'entraînement / évaluation
### 4.1 Le MLP
`[Linear → (BatchNorm) → activation → (Dropout)] × n → Linear`. On peut donner une activation par couche cachée
(P3.2). `features` est la dernière couche cachée, utilisée pour la t-SNE (partie 6).

In [ ]:
ACTIVATIONS = {"relu": nn.ReLU, "sigmoid": nn.Sigmoid, "tanh": nn.Tanh}

class MLP(nn.Module):
    def __init__(self, in_dim, hidden, n_classes=6, activation="relu", dropout=0.0, norm=None):
        super().__init__()
        acts = [activation] * len(hidden) if isinstance(activation, str) else list(activation)
        layers, prev = [], in_dim
        for h, a in zip(hidden, acts):
            layers.append(nn.Linear(prev, h))
            if norm == "batch":
                layers.append(nn.BatchNorm1d(h))
            layers.append(ACTIVATIONS[a]())
            if dropout > 0:
                layers.append(nn.Dropout(dropout))
            prev = h
        self.features = nn.Sequential(*layers)
        self.head = nn.Linear(prev, n_classes)

    def forward(self, x):
        return self.head(self.features(x))

def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

ARCHS = {"[64]": [64], "[128]": [128], "[256]": [256],
         "2 couches": [128, 64], "3 couches": [256, 128, 64], "4 couches": [256, 128, 64, 32]}
pd.Series({k: count_params(MLP(IN_DIM, h)) for k, h in ARCHS.items()}, name="paramètres").to_frame()

### 4.2 `train_epoch`, `evaluate`, `fit` (comme au TP2)
`fit` garde l'historique par époque ; avec `patience`, il fait de l'arrêt anticipé et restaure les poids de la
meilleure époque (F1 macro de validation). `run_seeds` répète l'entraînement sur les 3 graines (partie 2.3).

In [ ]:
def compute_metrics(y_true, y_pred):
    return {"accuracy": float(accuracy_score(y_true, y_pred)),
            "f1_macro": float(f1_score(y_true, y_pred, average="macro", zero_division=0))}

def make_optimizer(name, params, lr, weight_decay=0.0):
    if name == "adam":
        return torch.optim.Adam(params, lr=lr, weight_decay=weight_decay)
    return torch.optim.SGD(params, lr=lr, momentum=0.9, weight_decay=weight_decay)

def train_epoch(model, loader, lossfn, optimizer, device=DEVICE):
    model.train()
    total_loss, y_true, y_pred = 0.0, [], []
    for X, y in loader:
        X, y = X.to(device), y.to(device)
        optimizer.zero_grad()
        logits = model(X)
        loss = lossfn(logits, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(y)
        y_true.append(y.cpu()); y_pred.append(logits.argmax(1).cpu())
    y_true, y_pred = torch.cat(y_true).numpy(), torch.cat(y_pred).numpy()
    return {"loss": total_loss / len(y_true), **compute_metrics(y_true, y_pred)}

@torch.no_grad()
def evaluate(model, loader, lossfn, device=DEVICE, return_preds=False):
    model.eval()
    total_loss, y_true, y_pred = 0.0, [], []
    for X, y in loader:
        X, y = X.to(device), y.to(device)
        logits = model(X)
        total_loss += lossfn(logits, y).item() * len(y)
        y_true.append(y.cpu()); y_pred.append(logits.argmax(1).cpu())
    y_true, y_pred = torch.cat(y_true).numpy(), torch.cat(y_pred).numpy()
    out = {"loss": total_loss / len(y_true), **compute_metrics(y_true, y_pred)}
    if return_preds:
        out["y_true"], out["y_pred"] = y_true, y_pred
    return out

def fit(model, train_loader, val_loader, optimizer, epochs=EPOCHS, patience=None, device=DEVICE):
    lossfn = nn.CrossEntropyLoss()
    model.to(device)
    hist = {k: [] for k in ["train_loss", "train_acc", "train_f1", "val_loss", "val_acc", "val_f1"]}
    best, best_state, best_epoch, wait = -np.inf, None, 0, 0
    t0 = time.time()
    for ep in range(1, epochs + 1):
        tr = train_epoch(model, train_loader, lossfn, optimizer, device)
        va = evaluate(model, val_loader, lossfn, device)
        for split, m in (("train", tr), ("val", va)):
            hist[f"{split}_loss"].append(m["loss"])
            hist[f"{split}_acc"].append(m["accuracy"])
            hist[f"{split}_f1"].append(m["f1_macro"])
        if va["f1_macro"] > best:                               # meilleure époque selon le F1 de validation
            best, best_epoch, wait = va["f1_macro"], ep, 0
            best_state = copy.deepcopy(model.state_dict())
        else:
            wait += 1
        if patience is not None and wait >= patience:           # arrêt anticipé
            break
    if patience is not None and best_state is not None:
        model.load_state_dict(best_state)
    hist["best_epoch"] = best_epoch
    hist["train_time_s"] = time.time() - t0
    return hist

def run_seeds(build_fn, seeds=SEEDS, keep_models=False, **fit_kwargs):
    """build_fn(seed) -> (model, train_loader, val_loader, optimizer). Renvoie les historiques et moyenne ± écart-type."""
    runs = []
    for s in seeds:
        set_seed(s)
        model, tl, vl, opt = build_fn(s)
        h = fit(model, tl, vl, opt, **fit_kwargs)
        i = h["best_epoch"] - 1 if fit_kwargs.get("patience") else -1
        run = {"seed": s, "history": h, "val_acc": h["val_acc"][i], "val_f1": h["val_f1"][i]}
        if keep_models:
            run["model"] = model
        runs.append(run)
    summary = {}
    for k in ("val_acc", "val_f1"):
        v = np.array([r[k] for r in runs])
        summary[k] = {"mean": float(v.mean()), "std": float(v.std(ddof=1)) if len(v) > 1 else 0.0}
    return {"runs": runs, "summary": summary}

def mean_std(values):
    v = np.asarray(values, dtype=float)
    return f"{v.mean():.4f} ± {(v.std(ddof=1) if len(v) > 1 else 0.0):.4f}"

### 4.3 Relire ou calculer une expérience
Chaque expérience de la partie 5 est une fonction ; `experience` relit son fichier de résultats s'il existe,
sinon lance le calcul et l'enregistre.

In [ ]:
def save_json(obj, path):
    conv = lambda o: o.item() if isinstance(o, (np.floating, np.integer)) else o.tolist()
    Path(path).write_text(json.dumps(obj, indent=2, default=conv))

def experience(name, compute):
    path = LOG_DIR / f"{name}.json"
    if RECALCULER or not path.exists():
        save_json(compute(), path)
    return json.loads(path.read_text())

def summarize(logs, keys=None):
    """Tableau moyenne ± écart-type (3 graines) à partir des résultats de run_seeds."""
    rows = []
    for k in (keys or logs):
        s = logs[k]["summary"]
        hs = [r["history"] for r in logs[k]["runs"]]
        rows.append({"config": k,
                     "F1 val": f"{s['val_f1']['mean']:.4f} ± {s['val_f1']['std']:.4f}",
                     "acc val": f"{s['val_acc']['mean']:.4f} ± {s['val_acc']['std']:.4f}",
                     "Δ train−val": round(np.mean([h["train_acc"][-1] for h in hs]) - s["val_acc"]["mean"], 3),
                     "temps (s)": round(np.mean([h["train_time_s"] for h in hs]), 1),
                     "f1_mean": s["val_f1"]["mean"], "f1_std": s["val_f1"]["std"]})
    return pd.DataFrame(rows)

def build_mlp(hidden, lr, activation="relu", dropout=0.0, norm=None, wd=0.0, batch_size=64):
    def build(seed):
        tl, vl = make_loaders(X_tr, y_tr, X_va, y_va, batch_size=batch_size, seed=seed)
        model = MLP(IN_DIM, hidden, activation=activation, dropout=dropout, norm=norm)
        return model, tl, vl, make_optimizer("adam", model.parameters(), lr=lr, weight_decay=wd)
    return build

## 5. Boucle d'entraînement principale
### 5.1 Largeur de la couche cachée (P3.1)
Largeurs 64, 128, 256 ; pas d'apprentissage dans {1e-4, 3e-4, 1e-3, 3e-3} ; Adam, 30 époques, 3 graines.
Pour chaque largeur, on garde le meilleur pas (F1 macro de validation).

In [ ]:
def p31_largeur():
    logs = {}
    for width in [64, 128, 256]:
        for lr in [1e-4, 3e-4, 1e-3, 3e-3]:
            logs[f"w{width}_lr{lr}"] = run_seeds(build_mlp([width], lr), epochs=EPOCHS)
            print("fait :", width, lr, flush=True)
    return logs

w = experience("p31_width_lr_grid", p31_largeur)
t31 = summarize(w)
t31["largeur"] = t31["config"].str.extract(r"w(\d+)_")[0].astype(int)
t31["lr"] = t31["config"].str.extract(r"lr([\d.e-]+)")[0].astype(float)
t31["paramètres"] = [count_params(MLP(IN_DIM, [x])) for x in t31["largeur"]]
best31 = t31.loc[t31.groupby("largeur")["f1_mean"].idxmax()]
display(best31[["largeur", "paramètres", "lr", "F1 val", "acc val", "Δ train−val", "temps (s)"]])
print(f"écart maximal de F1 entre largeurs : {best31['f1_mean'].max() - best31['f1_mean'].min():.4f} "
      f"(écarts-types : {best31['f1_std'].min():.4f} à {best31['f1_std'].max():.4f})")

**Lecture.** Passer de 64 à 256 neurones multiplie les paramètres par quatre sans changer le F1 de validation de
façon détectable (écart inférieur aux écarts-types entre graines), alors que l'écart apprentissage − validation
augmente : un réseau plus large sur-apprend davantage sans mieux généraliser. Le meilleur pas est le plus petit de la
grille pour 64 et 128 : avec 30 époques, un petit pas joue en partie le rôle d'un arrêt anticipé.

### 5.2 Profondeur et combinaisons d'activations (P3.2)
2, 3 et 4 couches ; activations ReLU, tanh, sigmoïde, et ReLU avec tanh sur la dernière couche cachée ;
pas dans {1e-4, 3e-4, 1e-3} ; 30 époques, 3 graines.

In [ ]:
DEPTHS = {"2 couches": [128, 64], "3 couches": [256, 128, 64], "4 couches": [256, 128, 64, 32]}
def activation_of(name, depth):
    return ["relu"] * (depth - 1) + ["tanh"] if name == "relu+tanh" else name   # tanh sur la dernière couche

def p32_profondeur():
    logs = {}
    for arch, hidden in DEPTHS.items():
        for act_name in ["relu", "tanh", "sigmoid", "relu+tanh"]:
            for lr in [1e-4, 3e-4, 1e-3]:
                build = build_mlp(hidden, lr, activation=activation_of(act_name, len(hidden)))
                logs[f"{arch}_{act_name}_lr{lr}"] = run_seeds(build, epochs=EPOCHS)
                print("fait :", arch, act_name, lr, flush=True)
    return logs

d = experience("p32_depth_activation", p32_profondeur)
t32 = summarize(d)
parts = t32["config"].str.extract(r"^(?P<profondeur>.+?)_(?P<activation>.+)_lr(?P<lr>[\d.e-]+)$")
t32 = pd.concat([parts, t32], axis=1)
t32["paramètres"] = [count_params(MLP(IN_DIM, DEPTHS[p])) for p in t32["profondeur"]]
best32 = t32.loc[t32.groupby(["profondeur", "activation"])["f1_mean"].idxmax()].sort_values("f1_mean", ascending=False)
display(best32[["profondeur", "activation", "lr", "paramètres", "F1 val", "Δ train−val", "temps (s)"]])
for p in DEPTHS:
    sub = best32[(best32.profondeur == p) & (best32.activation != "sigmoid")]
    print(f"{p} : écart de F1 entre activations (hors sigmoïde) = {sub.f1_mean.max() - sub.f1_mean.min():.4f}, "
          f"écarts-types {sub.f1_std.min():.4f} à {sub.f1_std.max():.4f}")

**Lecture.** Ajouter des couches n'améliore pas la validation (meilleur F1 : 2 couches > 3 > 4), avec un écart
d'apprentissage et un temps croissants. À profondeur donnée, les activations ne se départagent pas (écarts du même
ordre que les écarts-types). Seul effet net : 4 couches avec sigmoïde n'apprend pas (F1 ≈ 0,18), vraisemblablement à
cause de l'atténuation des gradients à travers des sigmoïdes empilées (explication non vérifiée).
Pour la suite, on retient [256, 128, 64] avec ReLU, ReLU, tanh (pas 3·10⁻⁴), défendable mais non démontré meilleur.

### 5.3 Régularisation (P4)
Sur [256, 128, 64] (ReLU, ReLU, tanh ; pas 3·10⁻⁴ ; 30 époques ; 3 graines), chaque technique isolément puis
combinée. Pour les lignes avec arrêt anticipé, scores et loss sont pris à la meilleure époque (choisie sur la
validation, donc légèrement optimistes).

In [ ]:
HIDDEN, ACT, LR = [256, 128, 64], ["relu", "relu", "tanh"], 3e-4
REGULARISATIONS = {
    "base (aucune)":           dict(),
    "dropout 0.1":             dict(dropout=0.1),
    "dropout 0.3":             dict(dropout=0.3),
    "dropout 0.5":             dict(dropout=0.5),
    "weight decay 1e-5":       dict(wd=1e-5),
    "weight decay 1e-4":       dict(wd=1e-4),
    "weight decay 1e-3":       dict(wd=1e-3),
    "batchnorm":               dict(norm="batch"),
    "early stopping (p=5)":    dict(patience=5),
    "dropout 0.3 + wd 1e-4":   dict(dropout=0.3, wd=1e-4),
    "batchnorm + dropout 0.3": dict(norm="batch", dropout=0.3),
    "bn + drop 0.3 + wd + ES": dict(norm="batch", dropout=0.3, wd=1e-4, patience=5),
}
print("paramètres :", count_params(MLP(IN_DIM, HIDDEN, activation=ACT)))

def p4_regularisation():
    logs = {}
    for name, cfg in REGULARISATIONS.items():
        build = build_mlp(HIDDEN, LR, activation=ACT, dropout=cfg.get("dropout", 0.0),
                          norm=cfg.get("norm"), wd=cfg.get("wd", 0.0))
        logs[name] = run_seeds(build, epochs=EPOCHS, patience=cfg.get("patience"))
        print("fait :", name, flush=True)
    return logs

reg = experience("p4_regularisation", p4_regularisation)
rows = []
for name, cfg in REGULARISATIONS.items():
    runs = reg[name]["runs"]
    idx = [(r["history"]["best_epoch"] - 1) if cfg.get("patience") else -1 for r in runs]
    train_acc = [r["history"]["train_acc"][i] for r, i in zip(runs, idx)]
    rows.append({"config": name,
                 "val_f1": mean_std([r["val_f1"] for r in runs]),
                 "val_acc": mean_std([r["val_acc"] for r in runs]),
                 "val_loss": round(float(np.mean([r["history"]["val_loss"][i] for r, i in zip(runs, idx)])), 4),
                 "train_acc": round(float(np.mean(train_acc)), 4),
                 "gap": round(float(np.mean(train_acc) - np.mean([r["val_acc"] for r in runs])), 4),
                 "best_epoch": round(float(np.mean([r["history"]["best_epoch"] for r in runs])), 1)})
display(pd.DataFrame(rows))
base = reg["base (aucune)"]["summary"]["val_f1"]
for k in reg:
    s = reg[k]["summary"]["val_f1"]
    verdict = "significatif" if abs(s["mean"] - base["mean"]) > s["std"] + base["std"] else "non significatif"
    print(f"{k:26s} ΔF1 vs base = {s['mean'] - base['mean']:+.4f} (s1+s2 = {s['std'] + base['std']:.4f}) -> {verdict}")

**Lecture.** Aucune technique seule (hors arrêt anticipé) n'améliore le F1 de façon détectable ; dropout 0,5,
BatchNorm et arrêt anticipé réduisent en revanche la loss de validation (le réseau est moins sûr de lui à accuracy
quasi constante). La combinaison BatchNorm + dropout 0,3 + weight decay + arrêt anticipé est la meilleure ; ses
variantes sans arrêt anticipé restent au niveau de la base, ce qui suggère que l'arrêt anticipé en est l'ingrédient
principal (hypothèse non isolée par nos essais).

### 5.4 Bonus : recherche automatique d'hyperparamètres (P7)
30 configurations tirées au hasard (architecture, activation, pas, dropout, weight decay, BatchNorm, taille de lot),
une graine (100) et arrêt anticipé ; puis ré-évaluation des 3 meilleures sur les graines habituelles, pour limiter
le biais de sélection.

In [ ]:
SEARCH_ARCHS = {"[64]": [64], "[256]": [256], "[128,64]": [128, 64], "[256,128,64]": [256, 128, 64]}

def p7_recherche_aleatoire():
    rng = np.random.default_rng(123)
    def sample():
        return dict(arch=str(rng.choice(list(SEARCH_ARCHS))),
                    act=str(rng.choice(["relu", "tanh", "relu+tanh"])),
                    lr=float(10 ** rng.uniform(-4, np.log10(3e-3))),
                    dropout=float(rng.choice([0.0, 0.1, 0.3, 0.5])),
                    wd=float(rng.choice([0.0, 1e-5, 1e-4, 1e-3])),
                    norm=str(rng.choice(["none", "batch"])),
                    batch_size=int(rng.choice([32, 64, 128])))
    def make_build(cfg):
        hidden = SEARCH_ARCHS[cfg["arch"]]
        return build_mlp(hidden, cfg["lr"], activation=activation_of(cfg["act"], len(hidden)), dropout=cfg["dropout"],
                         norm=None if cfg["norm"] == "none" else cfg["norm"], wd=cfg["wd"], batch_size=cfg["batch_size"])
    # 1) recherche : une seule graine par essai, arrêt anticipé (patience 5)
    trials = []
    for i in range(30):
        cfg = sample()
        r = run_seeds(make_build(cfg), seeds=[100], epochs=30, patience=5)["runs"][0]
        trials.append({**cfg, "val_f1": r["val_f1"], "val_acc": r["val_acc"],
                       "best_epoch": r["history"]["best_epoch"], "time_s": round(r["history"]["train_time_s"], 1)})
        print(f"essai {i + 1}/30 : F1 {r['val_f1']:.4f}", flush=True)
    table = pd.DataFrame(trials).sort_values("val_f1", ascending=False).reset_index(drop=True)
    # 2) ré-évaluation des 3 meilleures sur les graines 0, 1, 2
    final = []
    for _, row in table.head(3).iterrows():
        cfg = {k: row[k] for k in ("arch", "act", "lr", "dropout", "wd", "norm", "batch_size")}
        cfg["batch_size"] = int(cfg["batch_size"])
        res = run_seeds(make_build(cfg), seeds=SEEDS, epochs=30, patience=5)
        final.append({**cfg, "params": count_params(make_build(cfg)(0)[0]),
                      "val_f1": mean_std([r["val_f1"] for r in res["runs"]]),
                      "val_acc": mean_std([r["val_acc"] for r in res["runs"]])})
    table.to_csv(RESULTS_DIR / "p7_random_search.csv", index=False)
    pd.DataFrame(final).to_csv(RESULTS_DIR / "p7_top3_reeval.csv", index=False)

if RECALCULER or not (RESULTS_DIR / "p7_top3_reeval.csv").exists():
    p7_recherche_aleatoire()
display(pd.read_csv(RESULTS_DIR / "p7_random_search.csv").head(5))
display(pd.read_csv(RESULTS_DIR / "p7_top3_reeval.csv"))

**Lecture.** Le meilleur score à une graine retombe une fois ré-évalué sur 3 graines (effet attendu de la sélection
parmi 30 essais) ; les trois meilleures configurations restent au niveau de la combinaison retenue en 5.3 : la
recherche aléatoire ne trouve rien de significativement meilleur.

### 5.5 Configurations retenues et contrôle de cohérence
Trois configurations sont retenues sur la validation : `combo` (BatchNorm + dropout 0,3 + weight decay + arrêt
anticipé), `es` (arrêt anticipé seul) et `dropout5` (dropout 0,5). On les **réentraîne** (mêmes graines, modèles
conservés pour le test) : les F1 de validation doivent redonner ceux de 5.3, ce qui vérifie que les modèles évalués
au test sont bien ceux qui ont été sélectionnés.

In [ ]:
SELECTED = {"combo": "bn + drop 0.3 + wd + ES", "es": "early stopping (p=5)", "dropout5": "dropout 0.5"}

final_runs = {}
for name, key in SELECTED.items():
    cfg = REGULARISATIONS[key]
    build = build_mlp(HIDDEN, LR, activation=ACT, dropout=cfg.get("dropout", 0.0),
                      norm=cfg.get("norm"), wd=cfg.get("wd", 0.0))
    final_runs[name] = run_seeds(build, keep_models=True, epochs=EPOCHS, patience=cfg.get("patience"))
    new, old = final_runs[name]["summary"]["val_f1"], reg[key]["summary"]["val_f1"]
    print(f"{name:9s} F1 val réentraîné {new['mean']:.4f} ± {new['std']:.4f} | 5.3 : {old['mean']:.4f} ± {old['std']:.4f} "
          f"| écart {abs(new['mean'] - old['mean']):.4f}")

## 6. Visualisation et comparaison des résultats
### 6.1 Courbes d'apprentissage (P5.1)

In [ ]:
def save_fig(fig, name):
    fig.savefig(FIG_DIR / f"{name}.png", dpi=200, bbox_inches="tight")

for name in SELECTED:
    h = final_runs[name]["runs"][0]["history"]
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    for ax, key, lab in zip(axes, ["loss", "acc", "f1"], ["Loss", "Accuracy", "F1 macro"]):
        ax.plot(h[f"train_{key}"], label="train"); ax.plot(h[f"val_{key}"], label="validation")
        ax.set_xlabel("époque"); ax.set_ylabel(lab); ax.grid(alpha=0.3); ax.legend()
    fig.suptitle(f"{name} (graine 0, meilleure époque = {h['best_epoch']})"); fig.tight_layout()
    save_fig(fig, f"v2_curves_{name}")

for key in ["val_f1", "val_loss"]:
    fig, ax = plt.subplots(figsize=(7, 4))
    for name in SELECTED:
        ax.plot(final_runs[name]["runs"][0]["history"][key], label=name)
    ax.set_xlabel("époque"); ax.set_ylabel(key); ax.grid(alpha=0.3); ax.legend(); fig.tight_layout()
    save_fig(fig, "v2_compare_f1" if key == "val_f1" else "v2_compare_loss")
plt.show()

# Stabilité : époque du minimum de la loss de validation et variabilité du F1 d'une époque à l'autre (sur la base de 5.3)
for name, key in {"base": "base (aucune)", **SELECTED}.items():
    vals = [(np.argmin(r["history"]["val_loss"]) + 1, np.min(r["history"]["val_loss"]),
             np.std(r["history"]["val_f1"][5:], ddof=1)) for r in reg[key]["runs"]]
    m = np.mean(vals, axis=0)
    print(f"{name:9s} époque du min. de loss {m[0]:5.1f} | min. loss val {m[1]:.4f} | écart-type du F1 (ép. ≥ 6) {m[2]:.4f}")

**Lecture.** Sans régularisation forte, la loss de validation remonte après quelques époques alors que le F1 reste
stable : le réseau devient trop sûr de ses prédictions. La combinaison garde la loss la plus basse. Aucune technique
ne réduit l'instabilité du F1 d'une époque à l'autre (dernier tableau).

### 6.2 t-SNE / UMAP de la dernière couche cachée (P6)
Fenêtres de **validation** (le test reste réservé à la section 7), modèle `combo`, graine 0. On compare les entrées
brutes (561 dimensions) et la dernière couche cachée (64 dimensions).

In [ ]:
@torch.no_grad()
def hidden_activations(model, X):
    model.eval()
    return model.features(torch.as_tensor(X, dtype=torch.float32, device=DEVICE)).cpu().numpy()

def project(Z, method="tsne", seed=0):
    if method == "tsne":
        return TSNE(n_components=2, perplexity=30, random_state=seed, init="pca", learning_rate="auto").fit_transform(Z)
    import umap
    return umap.UMAP(n_components=2, random_state=seed).fit_transform(Z)

def scatter_2d(Z, y, title, ax):
    for i, c in enumerate(CLASSES):
        ax.scatter(Z[y == i, 0], Z[y == i, 1], s=4, alpha=0.6, label=c)
    ax.set_title(title); ax.set_xticks([]); ax.set_yticks([]); ax.legend(markerscale=4, fontsize=8)

final_model = final_runs["combo"]["runs"][0]["model"]
H_va = hidden_activations(final_model, X_va)
try:
    import umap  # noqa: F401
    methods = ["tsne", "umap"]
except ImportError:
    methods = ["tsne"]                                         # UMAP facultatif (pip install umap-learn)
for method in methods:
    fig, ax = plt.subplots(1, 2, figsize=(14, 6))
    scatter_2d(project(X_va, method), y_va, f"{method.upper()} — entrées brutes ({X_va.shape[1]} dim.)", ax[0])
    scatter_2d(project(H_va, method), y_va, f"{method.upper()} — dernière couche cachée", ax[1])
    fig.tight_layout(); save_fig(fig, f"v2_{method}"); plt.show()

# Séparation des classes, mesurée sans passer par la figure : silhouette et pureté des 10 plus proches voisins
def knn_purity(Z, y, k=10):
    """Part des k plus proches voisins de même classe, moyennée sur les 6 classes."""
    idx = NearestNeighbors(n_neighbors=k + 1).fit(Z).kneighbors(Z, return_distance=False)[:, 1:]
    same = y[idx] == y[:, None]
    return float(np.mean([same[y == i].mean() for i in range(len(CLASSES))]))

pd.DataFrame({name: {"silhouette": silhouette_score(Z, y_va), "pureté 10-NN": knn_purity(Z, y_va)}
              for name, Z in {"entrées": X_va, "couche cachée (combo)": H_va}.items()}).T.round(3)

**Lecture.** Dans la couche cachée, les activités dynamiques et « allongé » forment des groupes séparés ; « assis » et
« debout » se recouvrent partiellement, en accord avec la confusion principale observée au test. Les distances entre
groupes et leurs tailles ne sont pas interprétables (elles dépendent de la perplexité et de la graine de la t-SNE).

## 7. Évaluation finale sur le jeu de test (P5.2) — une seule fois
Configurations figées (section 5.5) ; c'est le **seul** chargement de `test.csv` de la version 2.
Pour chaque configuration : F1 de validation et de test sur les 3 graines, F1 par classe, et matrice de confusion
cumulée sur les 3 graines, normalisée par ligne. (La référence et la V1 sont évaluées dans la V1.)

In [ ]:
df_test = pd.read_csv(DATA_DIR / "test.csv")
X_te_raw, y_te = to_xy(df_test)
X_te = scaler.transform(X_te_raw).astype(np.float32)          # scaler de l'apprentissage
test_loader = DataLoader(TensorDataset(torch.as_tensor(X_te), torch.as_tensor(y_te)), batch_size=512)
lossfn = nn.CrossEntropyLoss()

rows, per_class, cms, test_results = [], {}, {}, {}
for name in SELECTED:
    evals = [evaluate(r["model"], test_loader, lossfn, return_preds=True) for r in final_runs[name]["runs"]]
    f1_te = [e["f1_macro"] for e in evals]; acc_te = [e["accuracy"] for e in evals]
    f1_va = final_runs[name]["summary"]["val_f1"]
    rows.append({"configuration": name, "F1 val": f"{f1_va['mean']:.4f} ± {f1_va['std']:.4f}",
                 "F1 test": mean_std(f1_te), "acc test": mean_std(acc_te),
                 "Δ test − val": round(np.mean(f1_te) - f1_va["mean"], 4)})
    per_class[name] = pd.DataFrame([dict(zip(CLASSES, f1_score(e["y_true"], e["y_pred"], average=None,
                                                               labels=range(6), zero_division=0))) for e in evals]).mean()
    cms[name] = sum(confusion_matrix(e["y_true"], e["y_pred"], labels=range(6)) for e in evals)
    test_results[name] = {"f1_test": f1_te, "acc_test": acc_te, "f1_val": f1_va,
                          "f1_par_classe": per_class[name].to_dict(), "confusion_cumulee": cms[name].tolist()}
display(pd.DataFrame(rows))
display(pd.DataFrame(per_class).round(3))
for name in SELECTED:
    cm = cms[name] / cms[name].sum(axis=1, keepdims=True)
    fig, ax = plt.subplots(figsize=(7, 6))
    sns.heatmap(cm, annot=True, fmt=".2f", cmap="Blues", xticklabels=CLASSES, yticklabels=CLASSES, ax=ax)
    ax.set_xlabel("prédit"); ax.set_ylabel("vrai"); ax.set_title(f"{name} — test (3 graines cumulées)")
    fig.tight_layout(); save_fig(fig, f"v2_cm_test_{name}")
plt.show()
save_json(test_results, RESULTS_DIR / "v2_test_results.json")

**Classes les plus difficiles.** « Assis » et « debout » ont les F1 les plus bas et se confondent mutuellement :
deux postures statiques dont les signaux d'accéléromètre et de gyroscope sont très proches, seule une faible
différence d'inclinaison du tronc les séparant (explication plausible, non vérifiée). Parmi les marches, « montée »
et « descente » se confondent entre elles et avec la marche, des mouvements périodiques voisins.

**Validation et test.** Les trois configurations perdent du F1 au test : la validation (5 volontaires) est optimiste.
Une partie peut venir de la sélection sur la validation (arrêt anticipé, choix des configurations), une autre du
changement de volontaires ; nos résultats ne permettent pas de séparer les deux effets.